# Trade Review Deep Agent — go/no-go on a $100K futures portfolio

Financial Portfolio Trade Review using the **Deep Agents** framework — LangChain's batteries-included
agent layer that sits on top of LangGraph — applied to **Project 3B: Multi-Agent Deal
Review Pipeline** from the Week 3 Agentic AI Systems handout, adapted to a financial
deal-review use case: deciding whether to take a specific futures trade.

We build a **trade review pipeline**: an orchestrator that delegates to specialist
subagents to research a contract, size a position, check it against hard risk rules, and
draft a trade ticket — then **a human makes the final call** before anything is marked
as taken.

> **Decision-support only.** This agent never places a real order. `execute_trade` is a
> **simulated** tool, gated behind human approval, and the whole system is for
> educational/informational purposes — see the disclaimer in `README.md`.

**The one-liner (per the Week 3 framework):**
> My agent helps a futures trader decide whether to take a proposed trade on a $100K
> portfolio, replacing the 15–20 minutes of manual cross-checking of technical bias, risk
> sizing, and compliance rules it takes today. It researches the contract's setup, sizes
> the position under the 1–2% risk cap, checks it against hard portfolio risk rules, and
> drafts a trade ticket on its own using 5 tools, hands off to the trader for final
> approval before any trade is marked "taken," and I'll know it works when a trader can
> get a go/no-go decision with full reasoning in under 2 minutes that they'd act on 8
> times out of 10.

## Architecture at a glance

```
                  ┌───────────────────────────────────┐
  Trade idea      │     Trade Review Orchestrator      │
      │           │   (plans, delegates, assembles)    │
      └──────────▶│   write_todos → task → execute_trade│
                  └───┬─────────┬─────────┬──────┬──────┘
                      │ task    │ task    │ task │ task
                 ┌────▼───┐ ┌───▼─────┐ ┌──▼────┐ ┌▼─────┐
                 │market- │ │risk-    │ │compli-│ │trade-│
                 │analyst │ │sizer    │ │ance-  │ │critic│
                 │        │ │         │ │checker│ │      │
                 │ web    │ │ risk_   │ │compli-│ │both  │
                 │ search │ │calculator│ │ance_  │ │tools │
                 │        │ │         │ │check  │ │      │
                 └────┬───┘ └───┬─────┘ └──┬────┘ └──┬───┘
                      └─────────┴────┬─────┴─────────┘
                            ┌────────▼─────────┐
                            │   SHARED STATE   │
                            │ files + todos    │  ← ticket passes between agents here
                            └──────────────────┘
 Persistence: disk (/research, /drafts, /review, /final) + Store (/memories — cross-session)
```

**Deep Agents concepts this notebook demonstrates:**

| Concept | Where you'll see it |
|---|---|
| **Planning** | `write_todos` breaks the trade idea into research → size → compliance → critique → decide |
| **Subagents** | `task` delegates to specialists, each with its own tools, prompt, and *isolated context* |
| **Shared state** | message histories are isolated, but the **filesystem + todos are shared** — the ticket passes between agents via files |
| **Skills** | `SKILL.md` style guides loaded on demand (progressive disclosure) |
| **Memory** | `/memories/` persists the risk profile across sessions via a `Store` |
| **Human-in-the-loop** | the `execute_trade` action pauses for the trader's go/no-go |
| **Runaway protection** | `recursion_limit` is the hard backstop on the agent loop |

![Diagram](./images/trade_01_img.png)

## 1. Install & API keys

Dependencies are managed by **uv** (see `pyproject.toml` — same environment as
`gtm_deep_agent.ipynb`). If you opened this notebook with the project's `.venv` kernel,
everything is already installed.

In [1]:
# Already handled by `uv sync`. Uncomment to install into the current kernel if needed:
#%pip install -q deepagents "langchain>=1.0,<2.0" langchain-openai langchain-tavily python-dotenv

In [1]:
import os, getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads .env if present

def _need(key, prompt):
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(prompt)

_need("OPENAI_API_KEY", "OpenAI API key: ")
# Tavily is OPTIONAL — press Enter to skip and run grounding-only.
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key (Enter to skip): ")

# LangSmith tracing is OPTIONAL — set LANGSMITH_API_KEY in .env to turn it on.
if os.environ.get("LANGSMITH_API_KEY", "").strip():
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "trade-deep-agent")
    os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
    print(f"LangSmith tracing: ENABLED → project '{os.environ['LANGSMITH_PROJECT']}'")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: disabled (no LANGSMITH_API_KEY in .env)")

# Per the Week 3 handout: both tracks must use Nebius Token Factory for at least one
# model call. If NEBIUS_API_KEY + NEBIUS_BASE_URL are set, the critic subagent below
# routes through Nebius via the OpenAI-compatible endpoint; otherwise everything uses
# the primary MODEL.
MODEL = "gpt-4.1-mini"
NEBIUS_MODEL = os.environ.get("NEBIUS_MODEL", "meta-llama/Llama-3.3-70B-Instruct")
USE_NEBIUS = bool(os.environ.get("NEBIUS_API_KEY", "").strip())
print("Model:", MODEL, "| Nebius critic model:", NEBIUS_MODEL if USE_NEBIUS else "disabled (no NEBIUS_API_KEY)")

LangSmith tracing: ENABLED → project 'trade-deep-agent'
Model: gpt-4.1-mini | Nebius critic model: disabled (no NEBIUS_API_KEY)


## 2. Portfolio grounding

We ground the agent in two files instead of one:

- **`trading_playbook.md`** — the hard risk caps (never-rules) and the May 31, 2026
  macro snapshot.
- **`trading_strategy_dashboard.md`** — the detailed strategy rulebook pulled from the
  *Financial Portfolio Intelligence System Dashboard* (entry rules, stop/exit rules,
  position sizing, strategy-review triggers, and the live per-contract grade scorecard).
  The trade-critic subagent validates every reviewed trade against this file specifically.

Grounding is how we keep an LLM from inventing risk rules, strategy rules, or market
facts — the agent is told to honor these files and not go beyond them.

In [2]:
from datetime import date

GROUNDING = Path("grounding/trading_playbook.md").read_text()
STRATEGY_RULES = Path("grounding/trading_strategy_dashboard.md").read_text()

# The playbook's macro snapshot is dated May 31, 2026 and will only get staler. The
# market-analyst subagent needs to know what day it actually is *right now* so it can
# tell a live search result apart from that fixed snapshot, and label its brief
# correctly instead of defaulting to the snapshot's date out of habit.
TODAY = date.today().strftime("%B %d, %Y")

print(GROUNDING)
print("\n" + "=" * 70 + "\n")
print(STRATEGY_RULES)
print(f"\n--> TODAY (used to date-stamp the market-analyst's brief): {TODAY}")

# Grounding: $100K Futures Portfolio — Trading Playbook

> Source: internal portfolio risk framework, anchored to the May 31, 2026 pre-market
> futures briefing already on file in this workspace. Brand/strategy context for the
> Trade Review Deep Agent. The agent must never invent rules beyond this file + research.

## Mandate
Manage a **$100,000** futures account targeting **>20% annual ROI** while keeping risk
of ruin near zero. Decisions are evaluated on risk-adjusted expectancy, not win rate —
a 2.7:1 average reward:risk has carried this account to +78% in simulation at only a
32.5% win rate. The job is never "predict the market," it is "size and filter every idea
so the good R:R survives and the bad ideas get rejected before capital is at risk."

## Contract universe (10 contracts, 4 sectors)
| Sector | Contracts |
|---|---|
| Equities | ES (S&P 500 E-Mini), NQ (Nasdaq 100 E-Mini), YM (Dow Jones E-Mini) |
| Metals | GC (Gold), SI (Silver) |
| Energy | CL (Crude Oil WTI), NG (Natur

## 3. Tools

Five tools, each teaching a distinct idea:

- **`web_search` (Tavily)** — the market-analyst reaching *outside itself* for fresh
  technical/macro context. Optional.
- **`portfolio_state`** — a deterministic lookup of current open positions and portfolio
  value (mocked here; swap for a real broker/portfolio API in production).
- **`risk_calculator`** — a deterministic tool that computes dollar risk, contract count,
  and R:R from entry/stop/target. Teaching point: the LLM **drafts the idea**, but plain
  code **does the arithmetic** — models are unreliable at exact position sizing.
- **`compliance_check`** — a deterministic pass/fail against the hard risk rules (single
  trade cap, portfolio heat cap, same-sector limit, minimum R:R, stop present).
- **`execute_trade`** — a **simulated** outbound action ("take the trade"), wrapped later
  in a human-approval gate. Nothing is ever sent to a real broker.

![Diagram](./images/trade_02_img.png)

In [3]:
from langchain.tools import tool

# --- Tool 1: Web search (external API) — optional ---------------------
research_tools = []
if os.environ.get("TAVILY_API_KEY", "").strip():
    from langchain_tavily import TavilySearch
    research_tools = [TavilySearch(max_results=3)]
    print("Web search: Tavily ENABLED")
else:
    print("Web search: disabled (no key) — market-analyst will rely on grounding")

# --- Tool 2: Mocked portfolio state (swap for a real broker API later) -
MOCK_PORTFOLIO = {
    "portfolio_value": 100000.0,
    "open_positions": [
        {"contract": "ES", "sector": "Equity", "direction": "LONG", "risk_pct": 1.5},
        {"contract": "GC", "sector": "Metals", "direction": "LONG", "risk_pct": 1.0},
    ],
}

@tool
def portfolio_state() -> str:
    """Return current portfolio value and open positions (mocked). Use before sizing a new trade."""
    import json
    return json.dumps(MOCK_PORTFOLIO, indent=2)

# --- Tool 3: Deterministic risk/sizing calculator -----------------------
@tool
def risk_calculator(entry: float, stop: float, target: float, portfolio_value: float,
                     conviction: str) -> str:
    """Compute dollar risk, reward:risk, and position-size guidance for a trade.
    conviction must be 'HIGH' or 'MEDIUM'. Risk % is 1.5% for HIGH, 1.0% for MEDIUM,
    capped at 2.0%."""
    conviction = conviction.upper().strip()
    risk_pct = 0.015 if conviction == "HIGH" else 0.010
    risk_pct = min(risk_pct, 0.02)
    risk_amt = round(portfolio_value * risk_pct, 2)
    stop_dist = abs(entry - stop)
    target_dist = abs(target - entry)
    if stop_dist == 0:
        return "ERROR: entry and stop cannot be equal — a trade must have a real stop distance."
    rr = round(target_dist / stop_dist, 2)
    direction = "LONG" if target > entry else "SHORT"
    return (
        f"risk_pct={risk_pct*100:.1f}% | risk_amt=${risk_amt:,.2f} | "
        f"stop_distance={stop_dist:.2f} | reward:risk={rr}:1 | implied_direction={direction}"
    )

# --- Tool 4: Deterministic compliance check ------------------------------
@tool
def compliance_check(risk_pct: float, rr: float, sector: str, open_positions_json: str,
                      has_stop: bool) -> str:
    """PASS/FAIL a trade against the hard risk rules. risk_pct as a percent (e.g. 1.5),
    rr as a ratio (e.g. 2.3), open_positions_json from portfolio_state, has_stop boolean."""
    import json
    issues = []
    if not has_stop:
        issues.append("No stop defined — automatic FAIL.")
    if risk_pct > 2.0:
        issues.append(f"Single-trade risk {risk_pct}% exceeds the 2.0% hard cap.")
    if rr < 2.0:
        issues.append(f"Reward:risk {rr}:1 is below the 2.0:1 minimum.")
    try:
        positions = json.loads(open_positions_json).get("open_positions", [])
    except Exception:
        positions = []
    same_sector = [p for p in positions if p.get("sector", "").lower() == sector.lower()]
    total_heat = sum(p.get("risk_pct", 0) for p in positions) + risk_pct
    if len(same_sector) >= 2:
        issues.append(f"Already {len(same_sector)} open position(s) in {sector} — sector limit is 2.")
    if total_heat > 6.0:
        issues.append(f"Portfolio heat would be {total_heat:.1f}% — exceeds the 6.0% cap.")
    return "PASS" if not issues else "FAIL:\n- " + "\n- ".join(issues)

# --- Tool 5: Mock trade execution (SIMULATION ONLY) ----------------------
@tool
def execute_trade(contract: str, direction: str, entry: float, stop: float, target: float,
                   size_contracts: int) -> str:
    """Mark a trade as TAKEN (SIMULATED — no real order is ever sent to a broker)."""
    return (
        f"[SIMULATED] Trade marked TAKEN: {direction} {size_contracts}x {contract} "
        f"@ {entry} | stop {stop} | target {target}. No real order was placed."
    )

print("Tools ready.")

Web search: Tavily ENABLED
Tools ready.


## 4. Skills (progressive disclosure)

A **skill** is a `SKILL.md` file with YAML frontmatter (`name` + `description`). The
agent only sees the *description* until it decides a skill is relevant — then it loads
the full body. This keeps the system prompt lean. Our two skills are the trade-ticket
format and the compliance-review format.

![Diagram](./images/trade_03_img.png)

In [4]:
print(Path("skills/trade-ticket-style/SKILL.md").read_text())
print("\n" + "=" * 70 + "\n")
print(Path("skills/compliance-review-style/SKILL.md").read_text())

---
name: trade-ticket-style
description: How to format a trade ticket/recommendation for the $100K futures portfolio — exact sizing, stop, target, and one-line rationale.
---

# Trade Ticket Style

## When to use
When drafting a specific trade recommendation (entry, stop, target, size) for one contract.

## Structure
1. **Header**: contract, sector, direction (LONG/SHORT), conviction (HIGH/MEDIUM).
2. **Levels**: entry zone, stop price, target price, reward:risk ratio.
3. **Sizing**: dollar risk amount, % of portfolio risked, contract count, resulting
   portfolio heat if this trade is added to current open positions.
4. **Rationale**: one tight paragraph — the technical/macro "why," grounded only in the
   research brief and the trading playbook. No invented facts.
5. **Risk check**: explicit confirmation the trade respects the 2% single-trade cap, the
   6% portfolio heat cap, and the same-sector position limit (or a flag if it doesn't).

## Voice
Direct, numbers-first. State dollar

## 5. Backends: files + cross-session memory

Two persistence layers, wired with a **`CompositeBackend`** that routes by path:

- **`/memories/*`** → `StoreBackend` — survives across sessions (the risk profile and
  flagged-issue history we learn once)
- **everything else** → `FilesystemBackend` on real disk — so `/research`, `/drafts`,
  `/review`, and `/final` show up in your file explorer during the demo

The **checkpointer** holds per-thread state and is *required* for the human-in-the-loop
gate on `execute_trade`.

![Diagram](./images/trade_04_img.png)

In [5]:
from deepagents.backends import CompositeBackend, FilesystemBackend, StoreBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

store = InMemoryStore()        # cross-session memory (use a PostgresStore in production)
checkpointer = MemorySaver()   # per-thread state + required for HITL interrupts

backend = CompositeBackend(
    default=FilesystemBackend(root_dir=".", virtual_mode=True),
    routes={"/memories/": StoreBackend(namespace=lambda ctx: ("filesystem",))},
)
print("Backend: disk (default) + Store (/memories/) | checkpointer + store ready")

Backend: disk (default) + Store (/memories/) | checkpointer + store ready


## 6. Specialist subagents

Each subagent is a dict: a name, a description (the orchestrator reads this to decide
*when* to delegate), a focused system prompt, its own tools, and its own skills.
Subagents are **stateless** — each `task` call is a fresh run — so instructions must be
complete, and (since subagent context is isolated from the orchestrator's) any grounding
text a subagent needs has to be embedded directly into its own prompt, not just
referenced.

Note the **division of labour**, mirroring the Week 3 "Multi-Agent Deal Review Pipeline"
framework (extract → check compliance → flag risk/summarize → orchestrate): the
market-analyst gets web search **and** both grounding files so it can read the *current*
market trend against the playbook's last snapshot, the risk-sizer only gets the
calculator, the compliance-checker only gets the compliance tool, and the critic gets
both review tools plus the strategy dashboard so it can validate the trade against the
actual entry/exit/sizing rules and the live per-contract grade — not just the high-level
risk caps.

![Diagram](./images/trade_05_img.png)

In [6]:
SKILLS = ["/skills/"]  # loaded from disk via the FilesystemBackend

market_analyst_subagent = {
    "name": "market-analyst",
    "description": (
        "Researches the CURRENT technical/macro trend for one futures contract and "
        "writes a short brief. Use first, before any sizing or compliance work."
    ),
    "system_prompt": f"""You research one futures contract for a proposed trade idea.
Today's actual date is {TODAY}. Your job is to read the CURRENT market trend, not just
repeat old grounding facts — and to be honest about which one you actually used.

1. If web_search is available, search for the contract's current price action, recent
   news, and macro catalysts (e.g. "{{contract}} futures technical outlook this week").
   Use it — this is how you get a trend read that is actually current, not the stale
   May 31, 2026 snapshot below. Whatever date your search results imply (today, or the
   article/publish date), use THAT date — never write "May 31, 2026" in your brief
   unless you are explicitly using the fallback snapshot and saying so.
2. If web_search is unavailable, fall back to the grounding snapshot below, and say so
   explicitly and unambiguously in your brief, e.g. "NO LIVE SEARCH — using the stale
   May 31, 2026 snapshot only; today is {TODAY}", so downstream reviewers know exactly
   how stale this read is.

Trading playbook grounding (hard risk rules + last known macro snapshot — never invent
facts beyond this + whatever you find live):
{GROUNDING}

Title your brief's header with the actual date of the information you used (today,
{TODAY}, for live search; or "May 31, 2026 snapshot" if you had no live search — never
both, never an unlabeled date). Write 5-8 crisp bullet points on the technical setup,
macro backdrop, how current your information is (live search vs. snapshot-only), and
your conviction call (HIGH or MEDIUM) to /research/market_brief.md, then return a
one-line summary including the conviction call and which date/source you used.""",
    "tools": research_tools,
}

risk_sizer_subagent = {
    "name": "risk-sizer",
    "description": (
        "Sizes a proposed trade (entry/stop/target) using risk_calculator and drafts a "
        "trade ticket. Use after market-analyst has set a conviction call."
    ),
    "system_prompt": (
        "Load the 'trade-ticket-style' skill and follow it strictly. Read "
        "/research/market_brief.md for the conviction call and rationale. Call "
        "portfolio_state to get current portfolio value and open positions, then call "
        "risk_calculator with the proposed entry/stop/target/conviction. Draft one trade "
        "ticket with the exact dollar risk, R:R, and contract count, then save it to "
        "/drafts/trade_ticket.md. Return the full ticket text."
    ),
    "tools": [portfolio_state, risk_calculator],
    "skills": SKILLS,
}

compliance_checker_subagent = {
    "name": "compliance-checker",
    "description": (
        "Checks a drafted trade ticket against the hard risk rules using compliance_check. "
        "Use after risk-sizer has produced /drafts/trade_ticket.md."
    ),
    "system_prompt": (
        "Load the 'compliance-review-style' skill and follow it strictly. Read "
        "/drafts/trade_ticket.md. Call portfolio_state, then call compliance_check with "
        "the ticket's risk_pct, reward:risk, sector, the open_positions_json from "
        "portfolio_state, and whether a stop is present. Save your PASS/FAIL review to "
        "/review/compliance_report.md. Return the verdict line plus the full review."
    ),
    "tools": [portfolio_state, compliance_check],
    "skills": SKILLS,
}

trade_critic_subagent = {
    "name": "trade-critic",
    "description": (
        "Cross-checks the trade ticket and compliance report for consistency, "
        "completeness, AND validates the trade against the trading strategy dashboard "
        "(entry/exit/sizing rules, live contract grade). Use last, before the "
        "orchestrator finalizes anything."
    ),
    "system_prompt": f"""Read /research/market_brief.md, /drafts/trade_ticket.md, and
/review/compliance_report.md. Review on four axes:

1. CONSISTENCY — do the ticket's numbers match what compliance_check actually
   evaluated? Re-run compliance_check yourself if anything looks off.
2. GROUNDING — is every claim in the ticket's rationale supported by the market brief
   or the trading playbook, with nothing invented? Also check the brief's own date
   label for self-consistency: if it claims a live search but the header still reads
   "May 31, 2026" (the playbook snapshot's date, not today's), that is a mislabeling
   bug — flag it explicitly as NEEDS REVISION (re-run market-analyst) rather than
   silently trusting a header that contradicts the body.
3. COMPLETENESS — does the ticket have an entry, stop, target, size, dollar risk, and
   R:R? Flag anything missing.
4. STRATEGY VALIDATION — check the trade against the trading strategy dashboard below.
   Specifically: is the contract's grade C or D (flag and recommend reduce-size/pause
   per the scorecard)? Does the position size match the dashboard's HIGH/MEDIUM sizing
   bands? Is the R:R at least 2:1? Note if the market brief's conviction call was based
   on a live search or a stale snapshot (less confidence if stale).

Trading strategy dashboard (validate the trade against this; never invent strategy
rules beyond it + the playbook):
{STRATEGY_RULES}

Write your full verdict — all four axes' findings, the READY FOR HUMAN REVIEW or
NEEDS REVISION line, and the D/C-grade flag if applicable — to /review/critic_verdict.md.
This file is what the human reviewer sees at the approval step, so make it complete and
self-contained: don't make the human go dig through the other files to understand why.

End with a short verdict line: READY FOR HUMAN REVIEW, or NEEDS REVISION with the specific
fix needed. If the contract is graded D, say so explicitly in the verdict even if you still
mark it READY FOR HUMAN REVIEW — the human makes the final call on D-graded contracts.""",
    "tools": [risk_calculator, compliance_check],
    "skills": SKILLS,
}

print("4 subagents defined.")

4 subagents defined.


## 7. Assemble the deep agent

This is the whole point of Deep Agents: you **configure**, you don't build the plumbing.
Planning, the filesystem tools, and delegation come for free.

![Diagram](./images/trade_06_img.png)

In [7]:
from deepagents import create_deep_agent

ORCHESTRATOR_PROMPT = f"""You are the trade review lead for a $100,000 futures
portfolio. For one proposed trade idea, you decide whether it is ready to take — but you
NEVER make the final call yourself; a human always approves or rejects before the trade
is marked taken.

Trading playbook grounding (always honor this; never invent rules or market facts beyond
it + research):
{GROUNDING}

Workflow — first call write_todos to plan, then:
1. Delegate to 'market-analyst' to research the CURRENT market trend for the contract
   and set a conviction call (writes /research/market_brief.md).
2. Delegate to 'risk-sizer' to size the trade and draft a ticket
   (writes /drafts/trade_ticket.md).
3. Delegate to 'compliance-checker' to PASS/FAIL it against the hard risk rules
   (writes /review/compliance_report.md).
4. Delegate to 'trade-critic' for a final cross-check AND strategy-dashboard validation
   (entry/exit/sizing rules + live contract grade). If it says NEEDS REVISION,
   re-delegate to the relevant subagent to fix it, then re-run the critic.
5. Once the critic says READY FOR HUMAN REVIEW, save the final ticket, compliance
   verdict, AND the critic's full verdict (from /review/critic_verdict.md) to
   /final/trade_ticket.md, so the human sees one consolidated file.
6. Save a one-paragraph note on this trade's risk profile and any flags to
   /memories/risk_profile.md for future sessions.
7. Finally, call execute_trade with the ticket's exact contract/direction/entry/stop/
   target/size. This call will pause for human approval — do not write any summary or
   final message until the human's decision comes back.

If compliance_check ever returns FAIL and the critic can't get it to PASS after one
revision, do NOT call execute_trade — instead report the FAIL and stop, recommending the
human reject the idea outright.

Keep everything concise and numbers-first."""

agent = create_deep_agent(
    model=MODEL,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[execute_trade],
    subagents=[market_analyst_subagent, risk_sizer_subagent, compliance_checker_subagent,
               trade_critic_subagent],
    backend=backend,
    store=store,
    skills=SKILLS,
    interrupt_on={"execute_trade": True},   # HITL gate — needs the checkpointer
    checkpointer=checkpointer,
)
print("Deep agent assembled.")

Deep agent assembled.


## 8. Run a trade idea

Before you type in a setup, we run a quick **market scan** across the watchlist (ES,
NQ, GC, CL, NG): the market-analyst subagent gives each one a CURRENT read (live search
when Tavily is enabled) and proposes a candidate setup that honors the strategy
dashboard's entry rules — or says no setup qualifies right now. These scans are
single-subagent, throwaway-thread calls; they never touch `/research`, `/drafts`, etc.
used by the real review pipeline below.

Then **you** pick the contract and enter the setup (symbol/entry/stop/target) you want
the full pipeline to research, size, and review. Watch the message stream on the real
run: you'll see the orchestrator's `task` calls and each subagent's **final report** —
but *not* the subagents' internal back-and-forth. That hidden churn is the
context-isolation benefit.

`recursion_limit` caps the agent loop — the hard backstop against runaway loops.

![Diagram](./images/trade_07_img.png)

In [8]:
# Clear scratch artifacts from any previous run so the demo starts clean.
# write_file refuses to overwrite an existing file (by design — it forces an
# explicit edit_file instead of silent clobbering), so stale /research, /drafts,
# /review, /final files from a prior run cause the orchestrator to burn steps on
# failed writes/edits and retries under new filenames — easily exhausting
# recursion_limit before the pipeline finishes. /memories/ is left alone since
# that's the intentional cross-session store.
import shutil
from langgraph.errors import GraphRecursionError

for d in ["research", "drafts", "review", "final"]:
    p = Path(d)
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

config = {"configurable": {"thread_id": "trade-demo-1"}, "recursion_limit": 60}

# --- Market scan: ask market-analyst for ONE candidate setup on each watched
# contract, grounded in the CURRENT market trend (web_search, when Tavily is
# enabled) plus the strategy dashboard's entry rules. Each scan runs on its own
# throwaway thread with a tight recursion limit so it never touches /research,
# /drafts, /review, /final used by the real review pipeline below.
WATCHLIST = ["ES", "NQ", "GC", "CL", "NG"]

def scan_contract(symbol: str) -> str:
    scan_config = {"configurable": {"thread_id": f"scan-{symbol}"}, "recursion_limit": 20}
    prompt = (
        "OVERRIDE — for this message only, skip your standing 7-step workflow. "
        f"ONLY delegate to 'market-analyst' to get a CURRENT read on {symbol} futures. "
        "Do not delegate to risk-sizer, compliance-checker, or trade-critic, and do not "
        "call execute_trade. Based on the market-analyst's brief and the strategy "
        "dashboard's entry rules (timeframe confirmation, MACD direction, RSI bounds, "
        "volume, no setups within 30 min of a major release), propose ONE candidate "
        "setup as a single line in exactly this format:\n"
        f"{symbol} | LONG-or-SHORT | entry=X | stop=X | target=X | HIGH-or-MEDIUM | "
        "one-line why\n"
        "If nothing qualifies right now, reply with exactly:\n"
        f"{symbol} | NO QUALIFYING SETUP | one-line why"
    )
    try:
        res = agent.invoke({"messages": [{"role": "user", "content": prompt}]}, config=scan_config)
        return res["messages"][-1].content
    except GraphRecursionError:
        return f"{symbol} | SCAN INCOMPLETE — recursion limit hit, size this one manually."

print("=" * 70)
print("MARKET SCAN — candidate setups across the watchlist")
print("=" * 70)
for sym in WATCHLIST:
    idea = scan_contract(sym)
    print(f"\n[{sym}]")
    print(idea)
print("\n" + "=" * 70)

# Enter the trade idea by hand, using the scan above as a guide. Press Enter on
# Symbol to fall back to the built-in NG example instead of typing all four.
DEFAULT_SYMBOL, DEFAULT_ENTRY, DEFAULT_STOP, DEFAULT_TARGET = "NG", 3.30, 3.18, 3.66

symbol = input(f"Symbol (e.g. ES, NQ, GC, CL, NG) [{DEFAULT_SYMBOL}]: ").strip().upper() or DEFAULT_SYMBOL

def _ask_price(label, default):
    raw = input(f"{label} [{default}]: ").strip()
    return float(raw) if raw else default

entry = _ask_price("Entry", DEFAULT_ENTRY)
stop = _ask_price("Stop", DEFAULT_STOP)
target = _ask_price("Target", DEFAULT_TARGET)

direction = "LONG" if target > entry else "SHORT"

trade_idea = (
    f"Trade idea: {direction} {symbol}, entry {entry}, stop {stop}, target {target}. "
    "Evaluate the current macro/technical backdrop from grounding and/or research, and "
    "decide HIGH or MEDIUM conviction yourself based on the setup."
)
print(f"\n--> Trade idea: {trade_idea}")

result = agent.invoke({"messages": [{"role": "user", "content": trade_idea}]}, config=config)
#try streaming the output to see how the agent is executing

for m in result["messages"]:
    m.pretty_print()


MARKET SCAN — candidate setups across the watchlist

[ES]
ES | SHORT | entry=4405 | stop=4425 | target=4365 | MEDIUM | MACD bearish divergence at all-time highs with neutral volume and no major releases within 30 min, targeting >2:1 R:R on cautious short.

[NQ]
NQ | SHORT | entry=18300 | stop=18420 | target=18060 | MEDIUM | MACD bearish divergence at all-time highs with medium conviction, targeting 2.0:1 R:R and tight stop below recent swing.

[GC]
GC | NO QUALIFYING SETUP | Consolidating below resistance with neutral MACD, no confirmed momentum or breakout, awaiting clearer directional confirmation.

[CL]
CL | SHORT | entry=79.80 | stop=81.20 | target=77.00 | MEDIUM | broke key $80 support on confirmed supply easing and bearish volume, target next support at $78.

[NG]
NG | NO QUALIFYING SETUP | Medium conviction with rebound potential but lacking clear MACD/RSI confirmation and near any major news release, no current entry rules fully met.


--> Trade idea: Trade idea: SHORT ES, entr

In [9]:
result

{'messages': [HumanMessage(content='Trade idea: SHORT ES, entry 4405.0, stop 4425.0, target 4367.0. Evaluate the current macro/technical backdrop from grounding and/or research, and decide HIGH or MEDIUM conviction yourself based on the setup.', additional_kwargs={}, response_metadata={}, id='4003364e-166c-4f73-80b9-4d445657ebc7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 7538, 'total_tokens': 7582, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 7040}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_108eae2bae', 'id': 'chatcmpl-DrRYYJfLb67VDwMWHMsXeH70HFcgl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed167-7dd6-7260-9e5b-284d68e46df7-0', tool_

![Diagram](./images/trade_08_img.png)

In [10]:
# Inspect the SHARED STATE the agents produced (the ticket passes between them via these files).
for path in ["research/market_brief.md", "drafts/trade_ticket.md",
             "review/compliance_report.md", "final/trade_ticket.md"]:
    p = Path(path)
    if p.exists():
        print(f"\n===== /{path} =====\n{p.read_text()}")

# The plan the orchestrator tracked:
print("\n===== TODOS =====")
for t in result.get("todos", []):
    print(f"[{t.get('status')}] {t.get('content')}")


===== /drafts/trade_ticket.md =====
Trade Ticket for SHORT ES

Entry: 4405
Stop: 4425
Target: 4367
Direction: SHORT
Conviction: MEDIUM
Max Risk: $1,000 (1% of $100k portfolio)
Stop Distance: 20 points
Reward:Risk Ratio: 1.9:1
Size: 1 contract

Rationale: The trade aligns with current macro and technical conditions indicating short bias. Risk is capped to 1% of portfolio, with a favorable risk:reward ratio of nearly 2:1.

===== TODOS =====
[in_progress] Delegate to market-analyst to research current macro/technical trend for ES with focus on the SHORT setup.


## 9. Human-in-the-loop: approve or reject taking the trade

We set `interrupt_on={"execute_trade": True}`, so the run **pauses** before marking the
trade taken and hands control back to us. Before asking for a decision, we print the
**trade-critic's full verdict** (`/review/critic_verdict.md`) — its consistency,
grounding, completeness, and strategy-dashboard checks, including any C/D contract-grade
flag — so the approve/reject call is grounded in the critic's actual reasoning, not just
the raw `execute_trade` arguments. We then **approve**, **reject with feedback**, or
**edit the arguments** before resuming with a `Command`. This is the non-negotiable
boundary from the trading playbook: reads (research, sizing, compliance checks) are
autonomous; the write action that "takes the trade" always needs a human.

![Diagram](./images/trade_09_img.png)

In [11]:
from langgraph.types import Command

# What is the agent waiting to do? In this LangChain version each interrupt's
# .value is a SINGLE HITLRequest dict: {"action_requests": [...], "review_configs": [...]}.
# One action_request == one hanging tool call.
def pending_requests(res):
    reqs = []
    for intr in res.get("__interrupt__", []):
        val = intr.value
        if isinstance(val, dict) and "action_requests" in val:
            reqs.extend(val["action_requests"])      # unwrap the HITLRequest
        elif isinstance(val, list):
            reqs.extend(val)                          # older list-style payload
        else:
            reqs.append(val)
    return reqs

# Surface the trade-critic's own verdict (axes + D/C-grade flags) so the human's
# accept/reject decision is grounded in *why* the agent thinks this is ready, not just
# the raw execute_trade arguments.
critic_verdict_path = Path("review/critic_verdict.md")
if critic_verdict_path.exists():
    print("=" * 70)
    print("TRADE-CRITIC VERDICT (strategy-dashboard validated)")
    print("=" * 70)
    print(critic_verdict_path.read_text())
    print("=" * 70 + "\n")
else:
    print("(No critic_verdict.md found — the critic may not have run, or the run "
          "stopped before it wrote one.)\n")

reqs = pending_requests(result)
if reqs:
    for r in reqs:
        print("PENDING APPROVAL — TAKE THIS TRADE?")
        print(r)
else:
    print("No interrupt pending (agent may have rejected the idea on compliance grounds, "
          "or finished without calling execute_trade).")


TRADE-CRITIC VERDICT (strategy-dashboard validated)
Trade-Critic Review Summary:

Verdict: NEEDS REVISION

Details:
- The trade ticket is consistent with sizing and risk controls for medium conviction (1% risk, 1 contract).
- Reward:risk ratio is 1.9:1, below the required minimum 2:1, violating the strategy hard rule.
- ES is graded B in the strategy dashboard, allowing trading, but the R:R is a minor breach.
- The rationale is generic and lacks detailed market evidence and timeframe confirmations.
- Compliance report confirms FAIL due to R:R violation.

Recommendation:
- Adjust trade parameters (target or stop) to achieve at least 2:1 reward:risk ratio.
- Enhance rationale with specific technical and macro confirmations.
- Resubmit for review.

This trade is NOT approved until revision is done.

No interrupt pending (agent may have rejected the idea on compliance grounds, or finished without calling execute_trade).


**This cell is the human decision point.** Running it will actually **prompt you for input** via `input()` — approve or reject right here in the notebook output, instead of setting a variable. In a real deployment this would be a UI action; here it is a real stdin prompt so the pause is genuinely interactive.

In [12]:
while "__interrupt__" in result:
    if critic_verdict_path.exists():
        print("=" * 70)
        print("TRADE-CRITIC VERDICT (strategy-dashboard validated)")
        print("=" * 70)
        print(critic_verdict_path.read_text())
        print("=" * 70)

    reqs = pending_requests(result)
    n = len(reqs)
    for r in reqs:
        print("PENDING APPROVAL — TAKE THIS TRADE?")
        print(r)

    raw = input("Approve this trade? [y]es / [n]o: ").strip().lower()
    decision = "approve" if raw in ("y", "yes", "") else "reject"
    print(f"--> Human decision: {decision.upper()}")

    if decision == "approve":
        decisions = [{"type": "approve"} for _ in range(n)]
    else:
        feedback = input("Reason for rejecting (sent back to the agent as feedback): ").strip()
        if not feedback:
            feedback = "Rejected by human reviewer — no reason given."
        print(f"--> Feedback sent to agent: {feedback}")
        decisions = [{"type": "reject", "message": feedback} for _ in range(n)]

    result = agent.invoke(Command(resume={"decisions": decisions}), config=config)

# Sanity check: confirm whether the trade was actually marked taken.
executed = [
    tc["args"]
    for m in result["messages"]
    for tc in (getattr(m, "tool_calls", None) or [])
    if tc["name"] == "execute_trade"
]
print("execute_trade calls made:", executed)

print("\n===== FINAL RESPONSE =====")
result["messages"][-1].pretty_print()


execute_trade calls made: []

===== FINAL RESPONSE =====
================================== Ai Message ==================================

The ES short trade at 4405 with stop at 4425 and target 4367 is rated MEDIUM conviction based on current macro easing and resistance near 4425. It risks $1,000 (1% of $100K) for 1 contract with a reward:risk ratio of 1.9:1, which fails the portfolio minimum of 2:1. Compliance check FAILs on this count. Trade-critic agrees it NEEDS REVISION due to the R:R shortfall and weak rationale grounding. I recommend adjusting target or stop to restore a minimum 2:1 R:R before considering human approval. Let me know if you want me to help revise or analyze alternate parameters.


## 10. Persistent memory across sessions

A **new thread** = a new session with a fresh message history. But anything written to
`/memories/` lives in the `Store`, not the thread — so a brand-new conversation can still
recall the risk profile and any flags raised on prior trades. This is the difference
between *thread state* and *long-term memory*.

![Diagram](./images/trade_10_img.png)

In [13]:
config2 = {"configurable": {"thread_id": "trade-demo-2"}, "recursion_limit": 20}
result2 = agent.invoke(
    {"messages": [{"role": "user",
                   "content": "Read /memories/risk_profile.md and summarise the current risk profile in one line."}]},
    config=config2,
)
result2["messages"][-1].pretty_print()

================================== Ai Message ==================================

Current risk profile: Medium conviction SHORT ES trade with $1,000 risk and 1.9:1 reward:risk, failing compliance due to insufficient minimum 2:1 R:R, moderate risk/reward discrepancy needing adjustment before approval.


## Recap

You built a multi-agent trade review pipeline with almost no plumbing code — the code
track's answer to the Week 3 "Multi-Agent Deal Review Pipeline" framework, applied to a
real $100K futures portfolio:

- An **orchestrator** that *planned* (`write_todos`) and *delegated* (`task`).
- Four **subagents** with isolated context and focused tools — research, sizing,
  compliance, and a final cross-check critic.
- **Skills** for trade-ticket and compliance-review format, loaded on demand.
- A **composite backend**: disk for the ticket/research/review artifacts, a Store for
  cross-session risk-profile memory.
- A **human-in-the-loop** gate on the only write action (`execute_trade`) — reads are
  autonomous, the trade decision is never.
- `recursion_limit` as the safety backstop.

**Where to take it next:** swap `InMemoryStore` for a `PostgresStore`, replace
`portfolio_state` with a real broker/portfolio API, add a position-correlation tool that
checks beyond same-sector overlap, or give the compliance-checker a structured
`response_format` so its verdict is machine-readable for a downstream dashboard.

**Disclaimer:** This notebook is for educational and informational purposes only. It is
not financial advice, and `execute_trade` never places a real order. Futures trading
involves substantial risk of loss — see the disclaimer already on file in
`Agentic-AI-Projects-main/Project_Vibe_Coding/`.